In [ ]:
from src.monitoring.feature_monitor import (
    calculate_feature_statistics
)

from src.monitoring.drift_detector import (
    detect_mean_drift
)

from src.monitoring.alert_generator import (
    generate_alerts
)

In [ ]:
SOURCE_TABLE = (
    "tep_anomaly.served.training_base"
)

df = spark.table(
    SOURCE_TABLE
)

In [ ]:
monitored_features = [
    "xmeas_1",
    "xmeas_2",
    "xmeas_3",
    "xmeas_1_delta1",
    "xmeas_2_delta1",
    "xmeas_3_delta1"
]


In [ ]:
baseline_stats = (
    calculate_feature_statistics(
        df,
        monitored_features
    )
)

In [ ]:
current_stats = (
    calculate_feature_statistics(
        df,
        monitored_features
    )
)

In [ ]:
current_stats["xmeas_1"]["mean"] *= 1.5

In [ ]:
drift_results = (
    detect_mean_drift(
        baseline_stats,
        current*stats,
        threshold=20
    )
)

drift_results

In [ ]:
alerts = generate_alerts(
   drift_results
)

alerts

In [ ]:
from pyspark.sql import Row

alert_rows = [
    Row(**alert)
    for alert in alerts
]

(
    spark.createDataFrame(alert_rows)
    .write
    .mode("append")
    .saveAsTable(
        "tep_anomaly.served.monitoring_alerts"
    )
)


In [ ]:
display(
    spark.sql(
        """
        SELECT *
        FROM tep_anomaly.served.monitoring_alerts
        ORDER BY alert_timestamp DESC
        """
    )
)